# TrashScan — Path B

Este notebook assume que a estrutura **já está baixada/criada no volume** do RunPod:

- `/workspace/.venv`
- `/workspace/TrashScan`
- `/workspace/TACO`
- `/workspace/external_datasets`
- `/workspace/processed_4cls`
- `/workspace/runs`

O foco aqui é rodar o fluxo do **Path B** usando os scripts `.py` do projeto, sem refazer downloads e sem refazer merge/preprocessamento por padrão.

## 1) Imports, paths e utilitários

In [ ]:
from pathlib import Path
import os
import sys
import shlex
import subprocess
import shutil
import torch
import pandas as pd
import json

# Caminhos principais no RunPod
WORKSPACE = Path('/workspace')
REPO_ROOT = WORKSPACE / 'TrashScan'

DATA_DIR = REPO_ROOT / 'data'
TRAIN_DIR = REPO_ROOT / 'train' / 'paths'
EVAL_DIR = REPO_ROOT / 'eval'

EXTERNAL_DIR = WORKSPACE / 'external_datasets'
TACO_DIR = WORKSPACE / 'TACO'
PROCESSED_DIR = WORKSPACE / 'processed_5cls'

DATASET_YAML_PATH_A = PROCESSED_DIR / 'dataset_path_A.yaml'
DATASET_YAML_PATH_B = PROCESSED_DIR / 'dataset_path_B.yaml'

# Path A: necessário porque o Path B usa o melhor detector treinado no Path A
RUNS_PATH_A_DIR = WORKSPACE / 'runs' / 'path_A'

# Path B: treino do classificador ViT-L
RUNS_PATH_B_DIR = WORKSPACE / 'runs' / 'path_B'
MLFLOW_DIR = Path('/root/mlflow')

# Resultados finais persistentes no volume
RESULTS_PATH_B_DIR = WORKSPACE / 'results_path_B'

# Scripts principais
TRAIN_PATH_B_SCRIPT = TRAIN_DIR / 'train_path_B.py'

for p in [RUNS_PATH_B_DIR, MLFLOW_DIR, RESULTS_PATH_B_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Python             =', sys.executable)
print('REPO_ROOT          =', REPO_ROOT)
print('TACO_DIR           =', TACO_DIR)
print('EXTERNAL_DIR       =', EXTERNAL_DIR)
print('PROCESSED_DIR      =', PROCESSED_DIR)
print('DATASET_YAML_PATH_A=', DATASET_YAML_PATH_A)
print('DATASET_YAML_PATH_B=', DATASET_YAML_PATH_B)
print('RUNS_PATH_A_DIR    =', RUNS_PATH_A_DIR)
print('RUNS_PATH_B_DIR    =', RUNS_PATH_B_DIR)
print('MLFLOW_DIR         =', MLFLOW_DIR)
print('RESULTS_PATH_B_DIR =', RESULTS_PATH_B_DIR)
print('TRAIN_PATH_B_SCRIPT=', TRAIN_PATH_B_SCRIPT)


def run_cmd(cmd, cwd=WORKSPACE, env=None):
    """Roda comandos de forma previsível no notebook."""
    if isinstance(cmd, str):
        print('$', cmd)
        return subprocess.run(cmd, cwd=str(cwd), shell=True, env=env, check=True)

    print('$', ' '.join(shlex.quote(str(x)) for x in cmd))
    return subprocess.run([str(x) for x in cmd], cwd=str(cwd), env=env, check=True)

Python             = /workspace/.venv/bin/python
REPO_ROOT          = /workspace/TrashScan
TACO_DIR           = /workspace/TACO
EXTERNAL_DIR       = /workspace/external_datasets
PROCESSED_DIR      = /workspace/processed_5cls
DATASET_YAML_PATH_A= /workspace/processed_5cls/dataset_path_A.yaml
DATASET_YAML_PATH_B= /workspace/processed_5cls/dataset_path_B.yaml
RUNS_PATH_A_DIR    = /workspace/runs/path_A
RUNS_PATH_B_DIR    = /workspace/runs/path_B
MLFLOW_DIR         = /root/mlflow
RESULTS_PATH_B_DIR = /workspace/results_path_B
TRAIN_PATH_B_SCRIPT= /workspace/TrashScan/train/paths/train_path_B.py


## 2) Verificação dos arquivos principais

In [3]:
required_paths = [
    TRAIN_PATH_B_SCRIPT,
    PROCESSED_DIR,
    RUNS_PATH_A_DIR,
]

missing = [str(p) for p in required_paths if not p.exists()]

if missing:
    print("Arquivos/pastas ausentes:")
    for p in missing:
        print(" -", p)
else:
    print("Tudo certo.")

Tudo certo.


## 3) Configuração de GPU e treino

In [ ]:
if torch.cuda.is_available():
    DEVICE = "0"
    gpu_name = torch.cuda.get_device_properties(0).name
else:
    DEVICE = "cpu"
    gpu_name = "cpu"

print("Device:", DEVICE)
print("GPU:", gpu_name)

EPOCHS = 100
BATCH = 8
LR = 5e-5
PATIENCE = 10
DET_CONF = 0.25

CLASSIFIERS = [
    "vit_l16_imagenet",
]

Device: 0
GPU: NVIDIA L4


## 4) Encontrar o melhor peso do Path A

In [ ]:
PATH_A_RUN_DIRS = [
    WORKSPACE / "runs" / "path_A",
    WORKSPACE / "runs" / "path_A_5cls",
    WORKSPACE / "runs" / "path_A_refined_head",
]

def read_yolo_results(run_dir: Path):
    """
    Lê métricas de uma pasta de treino YOLO.
    Espera estrutura:
      run_dir/
        weights/best.pt
        results.csv
        args.yaml
    """
    best_pt = run_dir / "weights" / "best.pt"
    results_csv = run_dir / "results.csv"
    metrics_json = run_dir / "metrics.json"

    if not best_pt.exists():
        return None

    row = {
        "group": run_dir.parent.name,
        "model": run_dir.name,
        "run_dir": run_dir,
        "best_pt": best_pt,
        "mAP50_95": None,
        "mAP50": None,
        "precision": None,
        "recall": None,
        "source": None,
    }

    if results_csv.exists():
        df = pd.read_csv(results_csv)
        df.columns = [c.strip() for c in df.columns]

        # Melhor época por mAP50-95, se existir
        map95_col = "metrics/mAP50-95(B)"
        map50_col = "metrics/mAP50(B)"
        precision_col = "metrics/precision(B)"
        recall_col = "metrics/recall(B)"

        if map95_col in df.columns:
            best_idx = df[map95_col].idxmax()
        elif map50_col in df.columns:
            best_idx = df[map50_col].idxmax()
        else:
            best_idx = df.index[-1]

        best = df.loc[best_idx]

        row["mAP50_95"] = float(best[map95_col]) if map95_col in df.columns else None
        row["mAP50"] = float(best[map50_col]) if map50_col in df.columns else None
        row["precision"] = float(best[precision_col]) if precision_col in df.columns else None
        row["recall"] = float(best[recall_col]) if recall_col in df.columns else None
        row["source"] = "results.csv"
        return row

    # Fallback para metrics.json, se existir
    if metrics_json.exists():
        with open(metrics_json, "r") as f:
            m = json.load(f)

        row["mAP50_95"] = m.get("mAP50_95")
        row["mAP50"] = m.get("mAP50")
        row["precision"] = m.get("precision")
        row["recall"] = m.get("recall")
        row["source"] = "metrics.json"
        return row

    # Tem best.pt, mas sem métrica
    row["source"] = "weights_only"
    return row


records = []

for base_dir in PATH_A_RUN_DIRS:
    if not base_dir.exists():
        print(f"[warn] Pasta não encontrada: {base_dir}")
        continue

    for run_dir in sorted(base_dir.iterdir()):
        if not run_dir.is_dir():
            continue

        rec = read_yolo_results(run_dir)
        if rec is not None:
            records.append(rec)

df_detectors = pd.DataFrame(records)

if df_detectors.empty:
    raise FileNotFoundError(
        "Nenhum detector com weights/best.pt foi encontrado em: "
        + ", ".join(str(p) for p in PATH_A_RUN_DIRS)
    )

# Ordena pelo melhor critério disponível
df_ranked = df_detectors.copy()
df_ranked["rank_score"] = df_ranked["mAP50_95"].fillna(df_ranked["mAP50"]).fillna(-1)

df_ranked = df_ranked.sort_values(
    by=["rank_score", "mAP50", "precision", "recall"],
    ascending=False,
    na_position="last",
).reset_index(drop=True)

display_cols = [
    "group", "model", "mAP50_95", "mAP50", "precision", "recall", "source", "best_pt"
]

display(df_ranked[display_cols])

best_detector = df_ranked.iloc[0]
DETECTOR_WEIGHTS = Path(best_detector["best_pt"])

print("Melhor detector encontrado:")
print("Grupo :", best_detector["group"])
print("Modelo:", best_detector["model"])
print("mAP50-95:", best_detector["mAP50_95"])
print("mAP50:", best_detector["mAP50"])
print("Pesos:", DETECTOR_WEIGHTS)

if not DETECTOR_WEIGHTS.exists():
    raise FileNotFoundError(f"Detector não encontrado: {DETECTOR_WEIGHTS}")

,group,model,mAP50_95,mAP50,precision,recall,source,best_pt
0,path_A_5cls,yolov11m_o2o,0.49421,0.71055,0.79748,0.64537,results.csv,/workspace/runs/path_A_5cls/yolov11m_o2o/weigh...
1,path_A_5cls,yolov11m,0.45791,0.66802,0.76878,0.61979,results.csv,/workspace/runs/path_A_5cls/yolov11m/weights/b...
2,path_A_5cls,yolov8m,0.45466,0.67461,0.81395,0.59336,results.csv,/workspace/runs/path_A_5cls/yolov8m/weights/be...
3,path_A,yolov11m,0.44551,0.64990,0.70430,0.58665,results.csv,/workspace/runs/path_A/yolov11m/weights/best.pt
4,path_A,yolov11m_o2o,0.44040,0.64164,0.68838,0.59821,results.csv,/workspace/runs/path_A/yolov11m_o2o/weights/be...
5,path_A,yolov10m,0.39211,0.57796,0.63463,0.53399,results.csv,/workspace/runs/path_A/yolov10m/weights/best.pt
6,path_A,yolov9s,0.38005,0.57062,0.65521,0.51124,results.csv,/workspace/runs/path_A/yolov9s/weights/best.pt
7,path_A_refined_head,yolov10n,0.04614,0.11337,0.18439,0.16914,results.csv,/workspace/runs/path_A_refined_head/yolov10n/w...
8,path_A,yolov8m,NaN,NaN,NaN,NaN,weights_only,/workspace/runs/path_A/yolov8m/weights/best.pt


Melhor detector encontrado:
Grupo : path_A_5cls
Modelo: yolov11m_o2o
mAP50-95: 0.49421
mAP50: 0.71055
Pesos: /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt


## 5) Conferir estrutura do Path B

In [8]:
for split in ["train", "val", "test"]:
    path_b_dir = PROCESSED_DIR / split / "path_B"
    print(split, path_b_dir, "->", path_b_dir.exists())

    for sub in ["images", "labels", "crops"]:
        p = path_b_dir / sub
        print("  ", sub, "->", p.exists())

train /workspace/processed_5cls/train/path_B -> True
   images -> True
   labels -> True
   crops -> True
val /workspace/processed_5cls/val/path_B -> True
   images -> True
   labels -> True
   crops -> True
test /workspace/processed_5cls/test/path_B -> True
   images -> True
   labels -> True
   crops -> True


## 6) Treino Path B — ViT-L fine-tuning

In [9]:
print("\nComando que será executado:")

print("\nParâmetros:")
print(f"Python executable: {sys.executable}")
print(f"Script: {TRAIN_PATH_B_SCRIPT}")
print(f"Detector weights: {DETECTOR_WEIGHTS}")
print(f"Crops dir: {PROCESSED_DIR}")
print(f"Output dir: {RUNS_PATH_B_DIR}")
print(f"Classifiers: {CLASSIFIERS}")
print(f"Epochs: {EPOCHS}")
print(f"Batch: {BATCH}")
print(f"Learning rate: {LR}")
print(f"Patience: {PATIENCE}")
print(f"Device: {DEVICE}")
print(f"Use YOLO crops: True")
print(f"Detection confidence: {DET_CONF}")
print(f"Crop cache dir: {RUNS_PATH_B_DIR / 'crop_cache'}")


Comando que será executado:

Parâmetros:
Python executable: /workspace/.venv/bin/python
Script: /workspace/TrashScan/train/paths/train_path_B.py
Detector weights: /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt
Crops dir: /workspace/processed_5cls
Output dir: /workspace/runs/path_B
Classifiers: ['vit_l16_imagenet']
Epochs: 100
Batch: 8
Learning rate: 5e-05
Patience: 10
Device: 0
Use YOLO crops: True
Detection confidence: 0.25
Crop cache dir: /workspace/runs/path_B/crop_cache


In [10]:
run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    "--detector_weights", str(DETECTOR_WEIGHTS),
    "--crops_dir", str(PROCESSED_DIR),
    "--output", str(RUNS_PATH_B_DIR),
    "--classifiers", *CLASSIFIERS,
    "--epochs", str(EPOCHS),
    "--batch", str(BATCH),
    "--lr", str(LR),
    "--patience", str(PATIENCE),
    "--device", str(DEVICE),
    "--use_yolo_crops",
    "--det_conf", str(DET_CONF),
    "--crop_cache_dir", str(RUNS_PATH_B_DIR / "crop_cache"),
])

$ /workspace/.venv/bin/python /workspace/TrashScan/train/paths/train_path_B.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --crops_dir /workspace/processed_5cls --output /workspace/runs/path_B --classifiers vit_l16_imagenet --epochs 100 --batch 8 --lr 5e-05 --patience 10 --device 0 --use_yolo_crops --det_conf 0.25 --crop_cache_dir /workspace/runs/path_B/crop_cache
Device : cuda:0
GPU    : NVIDIA L4  VRAM: 23.7GB
Class weights: {'plastic': 0.511, 'paper': 1.098, 'metal': 1.125, 'glass': 1.777, 'other': 0.489}

  Mode: YOLO on-the-fly crop extraction
  YOLO TTA: False
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
  [train] Loading YOLOCropDataset from cache: /workspace/runs/path_B/crop_cache/trai

  Built vit_l16_imagenet (vit_large_patch16_224.augreg_in21k_ft_in1k)  pretrained=True  303.3M params


  Ep   1/100 | train loss=0.7206 acc=0.7747 | val loss=0.3773 acc=0.8680 f1=0.8262


  Ep   2/100 | train loss=0.4474 acc=0.8527 | val loss=0.3472 acc=0.8802 f1=0.8455


  Ep   3/100 | train loss=0.3533 acc=0.8840 | val loss=0.3294 acc=0.8955 f1=0.8714


...

  Ep  21/100 | train loss=0.1495 acc=0.9442 | val loss=0.4463 acc=0.9148 f1=0.9040
  Early stopping at epoch 21 (best epoch 11, val_acc=0.9209)


  Test: 100%|██████████| 282/282 [00:38<00:00,  7.35it/s]



  [vit_l16_imagenet]  accuracy=0.9166  f1=0.9088  latency=22.48ms


Traceback (most recent call last):
  File "/workspace/TrashScan/train/paths/train_path_B.py", line 697, in <module>
    mlflow.log_metrics(
  File "/workspace/.venv/lib/python3.11/site-packages/mlflow/tracking/fluent.py", line 875, in log_metrics
    return MlflowClient().log_batch(
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspace/.venv/lib/python3.11/site-packages/mlflow/tracking/client.py", line 1847, in log_batch
    return self._tracking_client.log_batch(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspace/.venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/client.py", line 731, in log_batch
    self.store.log_batch(run_id=run_id, metrics=metrics_batch, params=[], tags=[])
  File "/workspace/.venv/lib/python3.11/site-packages/mlflow/store/tracking/file_store.py", line 1070, in log_batch
    metrics, params, tags = _validate_batch_log_data(metrics, params, tags)
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/wo

CalledProcessError: Command '['/workspace/.venv/bin/python', '/workspace/TrashScan/train/paths/train_path_B.py', '--detector_weights', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt', '--crops_dir', '/workspace/processed_5cls', '--output', '/workspace/runs/path_B', '--classifiers', 'vit_l16_imagenet', '--epochs', '100', '--batch', '8', '--lr', '5e-05', '--patience', '10', '--device', '0', '--use_yolo_crops', '--det_conf', '0.25', '--crop_cache_dir', '/workspace/runs/path_B/crop_cache']' returned non-zero exit status 1.

## Com TTA

In [ ]:
run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    "--detector_weights", str(DETECTOR_WEIGHTS),
    "--crops_dir", str(PROCESSED_DIR),
    "--output", str(RUNS_PATH_B_DIR),
    "--classifiers", *CLASSIFIERS,
    "--epochs", str(EPOCHS),
    "--batch", str(BATCH),
    "--lr", str(LR),
    "--patience", str(PATIENCE),
    "--device", str(DEVICE),
    "--use_yolo_crops",
    "--det_conf", str(DET_CONF),
    "--tta",
    "--crop_cache_dir", str(RUNS_PATH_B_DIR / "crop_cache_tta"),
])

In [ ]:
run_dir = Path("/workspace/runs/path_B/vit_l16_imagenet")

print("best.pt:", (run_dir / "weights" / "best.pt").exists())
print("history.csv:", (run_dir / "history.csv").exists())
print("metrics.json:", (run_dir / "metrics.json").exists())

if (run_dir / "metrics.json").exists():
    print(json.loads((run_dir / "metrics.json").read_text()))

if (run_dir / "history.csv").exists():
    hist = pd.read_csv(run_dir / "history.csv")
    display(hist.tail())

best.pt: True
history.csv: True
metrics.json: True
{'classifier': 'vit_l16_imagenet', 'accuracy': 0.91656, 'precision': 0.8934, 'recall': 0.93024, 'f1': 0.90878, 'latency_ms': 22.485, 'AP_plastic': 0.96898, 'AP_paper': 0.93228, 'AP_metal': 0.85618, 'AP_glass': 0.90456, 'AP_other': 0.96857}


,epoch,train_loss,train_acc,val_loss,val_acc,val_f1,lr
16,17,0.15920,0.94032,0.39439,0.92045,0.91331,0.000047
17,18,0.15638,0.94093,0.38688,0.91346,0.89121,0.000046
18,19,0.16158,0.93988,0.42557,0.91302,0.89447,0.000046
19,20,0.15993,0.94127,0.39342,0.91914,0.90987,0.000045
20,21,0.14946,0.94423,0.44625,0.91477,0.90401,0.000045


## 7) Resumo dos treinos

In [19]:
def run_cmd(cmd, cwd="/workspace", env=None, shell=False):
    print("$", " ".join(shlex.quote(str(x)) for x in cmd))
    if shell:
        return subprocess.run(cmd, cwd=str(cwd), shell=True, env=env, check=True)
    return subprocess.run([str(x) for x in cmd], cwd=str(cwd), env=env, check=True)
   
TRAIN_PATH_B_SCRIPT = Path("/workspace/TrashScan/train/paths/train_path_B.py")
RUNS_PATH_B_DIR = Path("/workspace/runs/path_B")
DETECTOR_WEIGHTS = Path("/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt")
PROCESSED_DIR = Path("/workspace/processed_5cls")

In [20]:
run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    "--detector_weights", str(DETECTOR_WEIGHTS),
    "--crops_dir", str(PROCESSED_DIR),
    "--output", str(RUNS_PATH_B_DIR),
    "--summarize",
])

$ /workspace/.venv/bin/python /workspace/TrashScan/train/paths/train_path_B.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --crops_dir /workspace/processed_5cls --output /workspace/runs/path_B --summarize



PATH B  —  CLASSIFIER BENCHMARK SUMMARY
                  accuracy  precision  recall     f1  latency_ms  AP_plastic  AP_paper  AP_metal  AP_glass  AP_other
classifier                                                                                                          
vit_l16_imagenet    0.9166     0.8934  0.9302 0.9088     22.4850      0.9690    0.9323    0.8562    0.9046    0.9686


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/train/paths/train_path_B.py', '--detector_weights', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt', '--crops_dir', '/workspace/processed_5cls', '--output', '/workspace/runs/path_B', '--summarize'], returncode=0)

## 8) Ver métricas do ViT-L

In [6]:
RUN_DIR = Path("/workspace/runs/path_B/vit_l16_imagenet")

history_path = RUN_DIR / "history.csv"
metrics_path = RUN_DIR / "metrics.json"
best_weights_path = RUN_DIR / "weights" / "best.pt"
cm_path = RUN_DIR / "confusion_matrix_vit_l16_imagenet.png"

print("Run dir:", RUN_DIR)
print("best.pt existe?", best_weights_path.exists())
print("history.csv existe?", history_path.exists())
print("metrics.json existe?", metrics_path.exists())
print("confusion matrix existe?", cm_path.exists())

if not history_path.exists():
    raise FileNotFoundError(f"history.csv não encontrado: {history_path}")

hist = pd.read_csv(history_path)

best_val_acc_idx = hist["val_acc"].idxmax()
best_val_f1_idx = hist["val_f1"].idxmax()

best_val_acc_row = hist.loc[best_val_acc_idx]
best_val_f1_row = hist.loc[best_val_f1_idx]

summary = {
    "run_dir": str(RUN_DIR),
    "best_weights": str(best_weights_path),
    "best_weights_exists": best_weights_path.exists(),

    "epochs_trained": int(hist["epoch"].max()),

    "best_epoch_by_val_acc": int(best_val_acc_row["epoch"]),
    "best_val_acc": float(best_val_acc_row["val_acc"]),
    "best_val_acc_val_f1": float(best_val_acc_row["val_f1"]),
    "best_val_acc_train_acc": float(best_val_acc_row["train_acc"]),
    "best_val_acc_train_loss": float(best_val_acc_row["train_loss"]),
    "best_val_acc_val_loss": float(best_val_acc_row["val_loss"]),

    "best_epoch_by_val_f1": int(best_val_f1_row["epoch"]),
    "best_val_f1": float(best_val_f1_row["val_f1"]),
    "best_val_f1_val_acc": float(best_val_f1_row["val_acc"]),
}

if metrics_path.exists():
    with open(metrics_path, "r") as f:
        test_metrics = json.load(f)

    summary.update({
        "test_accuracy": test_metrics.get("accuracy"),
        "test_precision_macro": test_metrics.get("precision"),
        "test_recall_macro": test_metrics.get("recall"),
        "test_f1_macro": test_metrics.get("f1"),
        "latency_ms": test_metrics.get("latency_ms"),
        "AP_plastic": test_metrics.get("AP_plastic"),
        "AP_paper": test_metrics.get("AP_paper"),
        "AP_metal": test_metrics.get("AP_metal"),
        "AP_glass": test_metrics.get("AP_glass"),
        "AP_other": test_metrics.get("AP_other"),
    })
else:
    print("\n⚠️ metrics.json não foi encontrado. Talvez o erro no MLflow tenha ocorrido antes de salvar as métricas.")

summary_df = pd.DataFrame([summary]).T.rename(columns={0: "value"})
display(summary_df)

if metrics_path.exists():
    print("\nMétricas de teste:")
    display(pd.DataFrame([test_metrics]))

Run dir: /workspace/runs/path_B/vit_l16_imagenet
best.pt existe? True
history.csv existe? True
metrics.json existe? True
confusion matrix existe? True


,value
run_dir,/workspace/runs/path_B/vit_l16_imagenet
best_weights,/workspace/runs/path_B/vit_l16_imagenet/weight...
best_weights_exists,True
epochs_trained,21
best_epoch_by_val_acc,11
best_val_acc,0.92089
best_val_acc_val_f1,0.91486
best_val_acc_train_acc,0.93128
best_val_acc_train_loss,0.19751
best_val_acc_val_loss,0.32909



Métricas de teste:


,classifier,accuracy,precision,recall,f1,latency_ms,AP_plastic,AP_paper,AP_metal,AP_glass,AP_other
0,vit_l16_imagenet,0.91656,0.8934,0.93024,0.90878,22.485,0.96898,0.93228,0.85618,0.90456,0.96857
